# MapEncoder Feature Map Visualization

Encodes occupancy maps with the trained `MapEncoder`, then maps the top-3 PCA components of the
16-d feature map to RGB — replicating Fig. 3 from *Joint Localization and Planning using Diffusion*
(Lao Beyer & Karaman, 2024).

Architecture reminder: `MapEncoder` = ResNet-18 first 3 layers (adapted for 1-channel input)
→ `[B, 128, H/8, W/8]` → 1×1 conv → `[B, 16, H/8, W/8]` (8×8 for 64×64 input).

In [ ]:
import sys, os
# Notebook lives in EllipsoidalCBFSampling/; project root is one level up
sys.path.insert(0, os.path.abspath(".."))

import torch
import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F
from sklearn.decomposition import PCA

from EllipsoidalCBFSampling.models.score_net_ellipsoids_rasterizedlocalfeats import TemporalUnet
from EllipsoidalCBFSampling.train_rasterizedlocalfeats import RasterizedEllipsoidsDataset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device: {device}')

## 1. Load checkpoint and extract MapEncoder

In [ ]:
CKPT = "checkpoints/rasterized_localfeats/ve_unet_largemaze_localfeats_step1000000.pt"

ckpt = torch.load(CKPT, map_location='cpu')
cfg  = ckpt['config']
print('Checkpoint step:', ckpt['step'])
print('Config:', {k: v for k, v in cfg.items() if k not in ('xy_min','xy_max','train_rasterized_dir','eval_rasterized_dir')})

model = TemporalUnet(
    state_dim=2,
    T_steps=cfg['T_steps'],
    unet_input_dim=cfg['unet_input_dim'],
    dim_mults=tuple(cfg['dim_mults']),
    local_dim=cfg['local_dim'],
)
model.load_state_dict(ckpt['score_net'])
model.eval()
model.to(device)

map_encoder = model.map_encoder
LOCAL_DIM   = cfg['local_dim']   # 16
print(f'MapEncoder loaded  (local_dim={LOCAL_DIM})')

## 2. Load occupancy maps from the training dataset

In [ ]:
TRAIN_DIR = "../Diffuser/TrajectoryDatasetGeneration/data/custom/largemaze_ellipsoids-v1_rasterized"

ds = RasterizedEllipsoidsDataset(TRAIN_DIR)

# Pick N maps spread evenly across the dataset for diversity
N = 16
indices  = torch.linspace(0, len(ds) - 1, N).long()
occ_maps = torch.stack([ds[i][1] for i in indices])   # [N, 1, H, W]
trajs    = torch.stack([ds[i][0] for i in indices])   # [N, T, 2]

H, W = occ_maps.shape[-2], occ_maps.shape[-1]
print(f'occ_maps: {occ_maps.shape}  (H={H}, W={W})')

## 3. Encode occupancy maps → spatial feature maps

In [ ]:
with torch.no_grad():
    feat_maps = map_encoder(occ_maps.to(device)).cpu()   # [N, 16, H/8, W/8]

print(f'feat_maps: {feat_maps.shape}')  # expect [16, 16, 8, 8]

## 4. PCA: 16-d features → 3 principal components → RGB

Fit PCA on all spatial locations across all N maps (`N × H' × W'` pixels, each a 16-d vector).
Project to 3-d and normalize each channel to [0, 1] for display.

In [ ]:
N_feat, C, Hf, Wf = feat_maps.shape

# [N, Hf, Wf, C] → [N*Hf*Wf, C]
pixels = feat_maps.permute(0, 2, 3, 1).reshape(-1, C).numpy()

pca  = PCA(n_components=3)
proj = pca.fit_transform(pixels)          # [N*Hf*Wf, 3]
rgb  = proj.reshape(N_feat, Hf, Wf, 3)   # [N, Hf, Wf, 3]

# Per-channel global min-max normalise to [0, 1]
for c in range(3):
    lo = rgb[..., c].min()
    hi = rgb[..., c].max()
    rgb[..., c] = (rgb[..., c] - lo) / (hi - lo + 1e-8)

print('Explained variance ratio (PC1, PC2, PC3):', pca.explained_variance_ratio_.round(3))
print(f'Total variance captured by top-3 PCs: {pca.explained_variance_ratio_.sum():.1%}')

## 5. Upsample feature maps to occupancy-map resolution (64×64)

In [ ]:
rgb_t  = torch.tensor(rgb, dtype=torch.float32).permute(0, 3, 1, 2)  # [N, 3, Hf, Wf]
rgb_up = F.interpolate(rgb_t, size=(H, W), mode='bilinear', align_corners=True)
rgb_up = rgb_up.permute(0, 2, 3, 1).numpy()   # [N, H, W, 3]

print(f'RGB upsampled: {rgb_up.shape}')

## 6. Visualization — replicating paper Fig. 3

Top row: occupancy maps (obstacles in blue).  
Bottom row: corresponding feature maps with top-3 PCA components mapped to RGB.

In [ ]:
cols = 8  # show first 8 examples
fig, axes = plt.subplots(2, cols, figsize=(cols * 2.5, 5.5))

for i in range(cols):
    ax_top = axes[0, i]
    ax_bot = axes[1, i]

    # Occupancy map
    ax_top.imshow(occ_maps[i, 0].numpy(), cmap='Blues', origin='lower', vmin=0, vmax=1)
    ax_top.set_xticks([]); ax_top.set_yticks([])
    if i == 0:
        ax_top.set_ylabel('Occupancy map', fontsize=9)

    # PCA RGB feature map (upsampled)
    ax_bot.imshow(rgb_up[i], origin='lower')
    ax_bot.set_xticks([]); ax_bot.set_yticks([])
    if i == 0:
        ax_bot.set_ylabel('Feature map\n(PCA → RGB)', fontsize=9)

plt.suptitle(
    f'MapEncoder feature maps — top-3 PCA components as RGB  '
    f'(ResNet-18 backbone → {C}-d → PCA 3)\n'
    f'Explained variance: PC1={pca.explained_variance_ratio_[0]:.1%}, '
    f'PC2={pca.explained_variance_ratio_[1]:.1%}, '
    f'PC3={pca.explained_variance_ratio_[2]:.1%}',
    fontsize=10
)
plt.tight_layout()
plt.savefig('feature_map_pca_visualization.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: feature_map_pca_visualization.png')

## 7. (Optional) Show all 16 individual feature channels

For one example, display each of the 16 channels of the raw feature map.

In [ ]:
EXAMPLE_IDX = 0
fm = feat_maps[EXAMPLE_IDX]   # [16, Hf, Wf]

fig, axes = plt.subplots(2, 8, figsize=(18, 5))
for ch, ax in enumerate(axes.flat):
    ax.imshow(fm[ch].numpy(), cmap='RdBu_r', origin='lower')
    ax.set_title(f'ch {ch}', fontsize=8)
    ax.axis('off')

fig.suptitle(f'All {C} raw feature channels  (example {EXAMPLE_IDX})', fontsize=11)
plt.tight_layout()
plt.savefig('feature_map_all_channels.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Local feature sampling visualization

Overlay sampled feature locations (trajectory waypoints) on the PCA RGB feature map,
showing exactly which feature vectors the model reads at each denoising step.

In [ ]:
from EllipsoidalCBFSampling.models.score_net_ellipsoids_rasterizedlocalfeats import sample_local_features

EXAMPLE_IDX = 0

occ  = occ_maps[EXAMPLE_IDX:EXAMPLE_IDX+1].to(device)   # [1, 1, H, W]
traj = trajs[EXAMPLE_IDX:EXAMPLE_IDX+1].to(device)       # [1, T, 2]

with torch.no_grad():
    fm_single = map_encoder(occ).cpu()              # [1, 16, Hf, Wf]
    local_f   = sample_local_features(fm_single, traj.cpu())  # [1, T, 16]

# PCA-project the sampled local features using the fitted PCA
T_len = traj.shape[1]
lf_proj = pca.transform(local_f[0].numpy())          # [T, 3]
# Re-normalise using the same global min/max from before
lf_rgb  = np.clip(lf_proj, proj.min(axis=0), proj.max(axis=0))
for c in range(3):
    lo = proj[:, c].min(); hi = proj[:, c].max()
    lf_rgb[:, c] = (lf_rgb[:, c] - lo) / (hi - lo + 1e-8)

# Waypoint pixel coords in [0, H] space  (positions are in [-1, 1])
wp = traj[0].cpu().numpy()                           # [T, 2]  (x, y) in [-1,1]
px = (wp[:, 0] + 1) / 2 * W                         # pixel col
py = (wp[:, 1] + 1) / 2 * H                         # pixel row (origin=lower)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))

axes[0].imshow(occ_maps[EXAMPLE_IDX, 0].numpy(), cmap='Blues', origin='lower', vmin=0, vmax=1)
axes[0].scatter(px, py, c=np.arange(T_len), cmap='plasma', s=12, zorder=3)
axes[0].set_title('Occupancy map + trajectory waypoints')
axes[0].axis('off')

axes[1].imshow(rgb_up[EXAMPLE_IDX], origin='lower')
axes[1].scatter(px, py, c=np.arange(T_len), cmap='plasma', s=12, zorder=3)
axes[1].set_title('PCA RGB feature map + sampled waypoints')
axes[1].axis('off')

plt.suptitle('Local feature sampling: waypoints coloured by position along trajectory\n'
             '(plasma: start=purple → end=yellow)', fontsize=10)
plt.tight_layout()
plt.savefig('local_feature_sampling.png', dpi=150, bbox_inches='tight')
plt.show()